In [4]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Load all tables ────────────────────────────────────────────────────────
cp = pd.read_csv('../data/customer_profile.csv')
ld = pd.read_csv('../data/loan_details.csv')
lo = pd.read_csv('../data/loan_outcome.csv')
rb = pd.read_csv('../data/repayment_behavior.csv')
rp = pd.read_csv('../data/risk_scored_portfolio.csv')

print("="*60)
print("LENDIITEAZY — VERIFIED NUMBERS FOR CRO REPORT")
print("="*60)

# ── PORTFOLIO OVERVIEW ─────────────────────────────────────────────────────
print("\n--- PORTFOLIO OVERVIEW ---")
print(f"Total loans              : {len(ld):,}")
print(f"Total customers          : {cp['customer_id'].nunique():,}")
print(f"Total defaults           : {lo['default_flag'].sum():,}")
print(f"Overall default rate     : {lo['default_flag'].mean()*100:.1f}%")
print(f"Total loan amount        : ₹{ld['loan_amount'].sum():,.0f}")
print(f"Total amount recovered   : ₹{lo['recovery_amount'].sum():,.0f}")
print(f"Total amount pending     : ₹{lo['total_amount_pending'].sum():,.0f}")
print(f"Total write offs         : {lo['write_off_flag'].sum():,}")
print(f"Recovery rate            : {(lo['recovery_amount'].sum()/ld['loan_amount'].sum())*100:.1f}%")

# ── EMPLOYMENT TYPE ────────────────────────────────────────────────────────
print("\n--- DEFAULT RATE BY EMPLOYMENT TYPE ---")
merged = ld.merge(lo[['loan_id','default_flag']], on='loan_id')
merged = merged.merge(cp[['customer_id','employment_type','city_tier',
                           'acquisition_channel','credit_score',
                           'monthly_income','age','existing_loans']], on='customer_id')
emp = merged.groupby('employment_type')['default_flag'].agg(
    loans='count', defaults='sum', default_rate='mean'
).reset_index()
emp['default_pct'] = (emp['default_rate']*100).round(1)
emp = emp.sort_values('default_pct', ascending=False)
for _, r in emp.iterrows():
    print(f"{r['employment_type']:15} : {r['default_pct']}%  ({int(r['defaults'])} defaults / {int(r['loans'])} loans)")

# ── PRODUCT TYPE ───────────────────────────────────────────────────────────
print("\n--- DEFAULT RATE BY PRODUCT TYPE ---")
prod = merged.groupby('product_type')['default_flag'].agg(
    loans='count', defaults='sum', default_rate='mean'
).reset_index()
prod['default_pct'] = (prod['default_rate']*100).round(1)
prod = prod.sort_values('default_pct', ascending=False)
for _, r in prod.iterrows():
    print(f"{r['product_type']:15} : {r['default_pct']}%  ({int(r['defaults'])} defaults / {int(r['loans'])} loans)")

print("\n--- AVERAGE LOAN AMOUNT BY PRODUCT ---")
print(ld.groupby('product_type')['loan_amount'].mean().round(0).to_string())

print("\n--- AVERAGE INTEREST RATE BY PRODUCT ---")
print(ld.groupby('product_type')['interest_rate'].mean().round(2).to_string())

# ── CITY TIER ──────────────────────────────────────────────────────────────
print("\n--- DEFAULT RATE BY CITY TIER ---")
city = merged.groupby('city_tier')['default_flag'].agg(
    loans='count', defaults='sum', default_rate='mean'
).reset_index()
city['default_pct'] = (city['default_rate']*100).round(1)
city = city.sort_values('default_pct', ascending=False)
for _, r in city.iterrows():
    print(f"{r['city_tier']:10} : {r['default_pct']}%  ({int(r['defaults'])} defaults / {int(r['loans'])} loans)")

# ── ACQUISITION CHANNEL ────────────────────────────────────────────────────
print("\n--- DEFAULT RATE BY ACQUISITION CHANNEL ---")
chan = merged.groupby('acquisition_channel')['default_flag'].agg(
    loans='count', defaults='sum', default_rate='mean'
).reset_index()
chan['default_pct'] = (chan['default_rate']*100).round(1)
chan = chan.sort_values('default_pct', ascending=False)
for _, r in chan.iterrows():
    print(f"{r['acquisition_channel']:15} : {r['default_pct']}%  ({int(r['defaults'])} defaults / {int(r['loans'])} loans)")

# ── RISK GRADE ─────────────────────────────────────────────────────────────
print("\n--- DEFAULT RATE BY RISK GRADE ---")
grade = merged.groupby('risk_grade')['default_flag'].agg(
    loans='count', defaults='sum', default_rate='mean'
).reset_index()
grade['default_pct'] = (grade['default_rate']*100).round(1)
grade = grade.sort_values('risk_grade')
for _, r in grade.iterrows():
    print(f"Grade {r['risk_grade']} : {r['default_pct']}%  ({int(r['defaults'])} defaults / {int(r['loans'])} loans)")

# ── CREDIT SCORE ───────────────────────────────────────────────────────────
print("\n--- CREDIT SCORE ANALYSIS ---")
d_avg  = merged[merged['default_flag']==1]['credit_score'].mean()
nd_avg = merged[merged['default_flag']==0]['credit_score'].mean()
print(f"Defaulter avg credit score     : {d_avg:.0f}")
print(f"Non defaulter avg credit score : {nd_avg:.0f}")
print(f"Gap                            : {nd_avg-d_avg:.0f} points")
print(f"Customers below 650            : {(cp['credit_score']<650).sum():,}  ({(cp['credit_score']<650).mean()*100:.1f}%)")
print(f"Customers below 600            : {(cp['credit_score']<600).sum():,}  ({(cp['credit_score']<600).mean()*100:.1f}%)")
print(f"Customers above 750            : {(cp['credit_score']>750).sum():,}  ({(cp['credit_score']>750).mean()*100:.1f}%)")

# ── INCOME AND EMI ─────────────────────────────────────────────────────────
print("\n--- INCOME BY EMPLOYMENT TYPE ---")
print(cp.groupby('employment_type')['monthly_income'].mean().round(0).to_string())

print("\n--- EMI TO INCOME RATIO ---")
emi_data = rp.groupby('employment_type')['emi_to_income'].mean().round(3)
print(emi_data.to_string())
print(f"\nAffordability stressed (EMI>40%) : {(rp['emi_to_income']>0.4).sum():,}  ({(rp['emi_to_income']>0.4).mean()*100:.1f}%)")
stressed_default = rp[rp['emi_to_income']>0.4]['default_flag'].mean()*100
normal_default   = rp[rp['emi_to_income']<=0.4]['default_flag'].mean()*100
print(f"Default rate EMI > 40%           : {stressed_default:.1f}%")
print(f"Default rate EMI <= 40%          : {normal_default:.1f}%")

# ── HEATMAP ────────────────────────────────────────────────────────────────
print("\n--- RISK HEATMAP (City Tier x Employment Type) ---")
heatmap = merged.pivot_table(
    values='default_flag',
    index='city_tier',
    columns='employment_type',
    aggfunc='mean'
)*100
print(heatmap.round(1).to_string())
print(f"\nHighest risk cell : {heatmap.max().max():.1f}%  ({heatmap.max().idxmax()} in {heatmap.max(axis=1).idxmax()})")
print(f"Lowest risk cell  : {heatmap.min().min():.1f}%  ({heatmap.min().idxmin()} in {heatmap.min(axis=1).idxmin()})")

# ── DPD EARLY WARNING ──────────────────────────────────────────────────────
print("\n--- DPD TREND ANALYSIS ---")
rb_merged = rb.merge(lo[['loan_id','default_flag']], on='loan_id')
dpd = rb_merged.groupby(['month_number','default_flag'])['dpd'].mean().reset_index()
defaulters     = dpd[dpd['default_flag']==1].set_index('month_number')['dpd']
non_defaulters = dpd[dpd['default_flag']==0].set_index('month_number')['dpd']
for m in [1,2,3,6,9,12]:
    if m in defaulters.index and m in non_defaulters.index:
        print(f"Month {m:2d}: Defaulters={defaulters[m]:.1f} DPD | Non-Defaulters={non_defaulters[m]:.1f} DPD | Gap={defaulters[m]-non_defaulters[m]:.1f}")

# ── PAYMENT STATUS EARLY MONTHS ────────────────────────────────────────────
print("\n--- PAYMENT STATUS MONTHS 1-3 FOR EVENTUAL DEFAULTERS ---")
early = rb_merged[(rb_merged['month_number']<=3) & (rb_merged['default_flag']==1)]
ps = early['payment_status'].value_counts(normalize=True)*100
print(ps.round(1).to_string())

print("\n--- BOUNCE RATE ANALYSIS ---")
bounce_d  = rb_merged[rb_merged['default_flag']==1]['bounce_flag'].mean()*100
bounce_nd = rb_merged[rb_merged['default_flag']==0]['bounce_flag'].mean()*100
print(f"Bounce rate defaulters     : {bounce_d:.1f}%")
print(f"Bounce rate non defaulters : {bounce_nd:.1f}%")

# ── RISK SCORECARD ─────────────────────────────────────────────────────────
print("\n--- SCORECARD DECISION ANALYSIS ---")
dec = rp.groupby('decision')['default_flag'].agg(
    loans='count', defaults='sum', default_rate='mean'
).reset_index()
dec['default_pct'] = (dec['default_rate']*100).round(1)
dec['pct_of_portfolio'] = (dec['loans']/len(rp)*100).round(1)
for _, r in dec.iterrows():
    print(f"{r['decision']:8} : {r['default_pct']}% default  ({int(r['loans'])} loans = {r['pct_of_portfolio']}% of portfolio)")

# ── RISK TIER ──────────────────────────────────────────────────────────────
print("\n--- RISK TIER DISTRIBUTION ---")
tier = rp.groupby('risk_tier')['default_flag'].agg(
    loans='count', defaults='sum', default_rate='mean'
).reset_index()
tier['default_pct'] = (tier['default_rate']*100).round(1)
for _, r in tier.iterrows():
    print(f"{r['risk_tier']:15} : {r['default_pct']}% default  ({int(r['loans'])} loans)")

# ── PRICING GAP ────────────────────────────────────────────────────────────
print("\n--- PRICING GAP BY EMPLOYMENT TYPE ---")
pricing = merged.groupby('employment_type').agg(
    avg_rate=('interest_rate','mean'),
    default_pct=('default_flag','mean')
).reset_index()
pricing['default_pct'] = pricing['default_pct']*100
pricing['rate_gap'] = pricing['default_pct'] - pricing['avg_rate']
pricing = pricing.round(2)
for _, r in pricing.iterrows():
    status = "UNDERPRICED" if r['rate_gap'] > 0 else "overpriced"
    print(f"{r['employment_type']:15} : Avg Rate={r['avg_rate']}%  Default={r['default_pct']}%  Gap={r['rate_gap']:+.1f}%  [{status}]")

print("\n--- PRICING GAP BY CITY TIER ---")
pricing_city = merged.groupby('city_tier').agg(
    avg_rate=('interest_rate','mean'),
    default_pct=('default_flag','mean')
).reset_index()
pricing_city['default_pct'] = pricing_city['default_pct']*100
pricing_city['rate_gap'] = pricing_city['default_pct'] - pricing_city['avg_rate']
pricing_city = pricing_city.round(2)
for _, r in pricing_city.iterrows():
    status = "UNDERPRICED" if r['rate_gap'] > 0 else "overpriced"
    print(f"{r['city_tier']:10} : Avg Rate={r['avg_rate']}%  Default={r['default_pct']}%  Gap={r['rate_gap']:+.1f}%  [{status}]")

# ── CO BORROWER ────────────────────────────────────────────────────────────
print("\n--- CO BORROWER ANALYSIS ---")
co = merged.groupby('co_borrower_flag')['default_flag'].mean()*100
print(f"With co-borrower    : {co.get(1, co.get('1', 'N/A')):.1f}%")
print(f"Without co-borrower : {co.get(0, co.get('0', 'N/A')):.1f}%")

# ── ACQUISITION CHANNEL SPLIT ──────────────────────────────────────────────
print("\n--- ACQUISITION CHANNEL CUSTOMER SPLIT ---")
ch_split = cp['acquisition_channel'].value_counts()
ch_pct   = cp['acquisition_channel'].value_counts(normalize=True)*100
for ch in ch_split.index:
    print(f"{ch:15} : {int(ch_split[ch]):,} customers  ({ch_pct[ch]:.1f}%)")

# ── EMPLOYMENT CUSTOMER SPLIT ──────────────────────────────────────────────
print("\n--- EMPLOYMENT TYPE CUSTOMER SPLIT ---")
emp_split = cp['employment_type'].value_counts()
emp_pct   = cp['employment_type'].value_counts(normalize=True)*100
for e in emp_split.index:
    print(f"{e:15} : {int(emp_split[e]):,} customers  ({emp_pct[e]:.1f}%)")

print("\n" + "="*60)
print("ALL NUMBERS VERIFIED FROM ACTUAL DATASET")
print("="*60)

LENDIITEAZY — VERIFIED NUMBERS FOR CRO REPORT

--- PORTFOLIO OVERVIEW ---
Total loans              : 12,000
Total customers          : 10,000
Total defaults           : 2,460
Overall default rate     : 20.5%
Total loan amount        : ₹2,813,368,111
Total amount recovered   : ₹97,324,797
Total amount pending     : ₹382,589,976
Total write offs         : 2,460
Recovery rate            : 3.5%

--- DEFAULT RATE BY EMPLOYMENT TYPE ---
Gig Worker      : 32.9%  (1377 defaults / 4183 loans)
Student         : 22.7%  (193 defaults / 850 loans)
Business Owner  : 20.9%  (163 defaults / 781 loans)
Self Employed   : 12.5%  (396 defaults / 3173 loans)
Salaried        : 11.0%  (331 defaults / 3013 loans)

--- DEFAULT RATE BY PRODUCT TYPE ---
BNPL            : 27.7%  (728 defaults / 2629 loans)
Education Loan  : 22.8%  (163 defaults / 715 loans)
SME Loan        : 18.9%  (366 defaults / 1939 loans)
Personal Loan   : 17.9%  (1203 defaults / 6717 loans)

--- AVERAGE LOAN AMOUNT BY PRODUCT ---
product_typ

In [5]:
# Run this after verify_numbers.py
# to confirm your three core numbers agree

total_loans    = len(ld)
default_rate   = lo['default_flag'].mean()*100
total_defaults = lo['default_flag'].sum()

calculated = round(total_loans * default_rate / 100)
print(f"Total loans    : {total_loans}")
print(f"Default rate   : {default_rate:.1f}%")
print(f"Total defaults : {total_defaults}")
print(f"Cross check    : {total_loans} x {default_rate:.1f}% = {calculated}")
print(f"Match          : {'YES' if calculated == total_defaults else 'NO - FIX THIS'}")

Total loans    : 12000
Default rate   : 20.5%
Total defaults : 2460
Cross check    : 12000 x 20.5% = 2460
Match          : YES


In [6]:
# Run this to verify your splits add up

print("Employment type % of LOANS:")
print((merged['employment_type'].value_counts(normalize=True)*100).round(1))
print(f"Sum: {merged['employment_type'].value_counts(normalize=True).sum()*100:.0f}%")

print("\nCity tier % of LOANS:")
print((merged['city_tier'].value_counts(normalize=True)*100).round(1))

print("\nChannel % of LOANS:")
print((merged['acquisition_channel'].value_counts(normalize=True)*100).round(1))

print("\nProduct % of LOANS:")
print((merged['product_type'].value_counts(normalize=True)*100).round(1))

Employment type % of LOANS:
employment_type
Gig Worker        34.9
Self Employed     26.4
Salaried          25.1
Student            7.1
Business Owner     6.5
Name: proportion, dtype: float64
Sum: 100%

City tier % of LOANS:
city_tier
Tier 2    46.0
Tier 3    28.8
Tier 1    25.2
Name: proportion, dtype: float64

Channel % of LOANS:
acquisition_channel
Social Media    34.7
Organic         30.1
Referral        20.0
DSA Agent       15.1
Name: proportion, dtype: float64

Product % of LOANS:
product_type
Personal Loan     56.0
BNPL              21.9
SME Loan          16.2
Education Loan     6.0
Name: proportion, dtype: float64


In [7]:
# Gig worker default rate must be the same
# whether from heatmap or bar chart

from_bar_chart = merged[merged['employment_type']=='Gig Worker']['default_flag'].mean()*100
from_heatmap   = merged[merged['employment_type']=='Gig Worker']['default_flag'].mean()*100

print(f"Bar chart gig worker rate : {from_bar_chart:.1f}%")
print(f"Heatmap gig worker avg    : {merged[merged['employment_type']=='Gig Worker'].groupby('city_tier')['default_flag'].mean().mean()*100:.1f}%")
print("Note: heatmap avg will differ slightly")
print("Always use bar chart number in report text")
print("Heatmap is for showing tier breakdown only")

Bar chart gig worker rate : 32.9%
Heatmap gig worker avg    : 32.4%
Note: heatmap avg will differ slightly
Always use bar chart number in report text
Heatmap is for showing tier breakdown only


In [8]:
# Print exact numbers Power BI should show
# Use these to verify your Power BI cards

print("NUMBERS TO VERIFY IN POWER BI:")
print(f"Total Loans card         : {len(ld):,}")
print(f"Total Customers card     : {cp['customer_id'].nunique():,}")
print(f"Default Rate card        : {lo['default_flag'].mean()*100:.1f}%")
print(f"Total Defaults card      : {lo['default_flag'].sum():,}")
print(f"Total Disbursed card     : ₹{ld['loan_amount'].sum():,.0f}")
print(f"Recovery Amount card     : ₹{lo['recovery_amount'].sum():,.0f}")
print(f"High Risk Loans card     : {(rp['risk_tier'].isin(['High Risk','Very High Risk'])).sum():,}")
print(f"Reject Count card        : {(rp['decision']=='REJECT').sum():,}")
print(f"Review Count card        : {(rp['decision']=='REVIEW').sum():,}")

NUMBERS TO VERIFY IN POWER BI:
Total Loans card         : 12,000
Total Customers card     : 10,000
Default Rate card        : 20.5%
Total Defaults card      : 2,460
Total Disbursed card     : ₹2,813,368,111
Recovery Amount card     : ₹97,324,797
High Risk Loans card     : 2,481
Reject Count card        : 5,913
Review Count card        : 3,525


In [9]:
# Quick final check script
checks = {
    "Total loans = 12000"              : len(ld) == 12000,
    "Total customers = 10000"          : len(cp) == 10000,
    "Default rate around 20.5%"        : abs(lo['default_flag'].mean()*100 - 20.5) < 1,
    "Gig worker highest default"       : merged.groupby('employment_type')['default_flag'].mean().idxmax() == 'Gig Worker',
    "BNPL highest product default"     : merged.groupby('product_type')['default_flag'].mean().idxmax() == 'BNPL',
    "Social media highest channel"     : merged.groupby('acquisition_channel')['default_flag'].mean().idxmax() == 'Social Media',
    "Tier 3 highest city default"      : merged.groupby('city_tier')['default_flag'].mean().idxmax() == 'Tier 3',
    "Referral lowest channel default"  : merged.groupby('acquisition_channel')['default_flag'].mean().idxmin() == 'Referral',
    "Salaried lowest employment default": merged.groupby('employment_type')['default_flag'].mean().idxmin() == 'Salaried',
    "5 risk_scored_portfolio columns"  : all(c in rp.columns for c in ['risk_score','risk_tier','decision','scorecard_points','risk_probability']),
}

print("FINAL CONSISTENCY CHECKS:")
all_passed = True
for check, result in checks.items():
    status = "PASS" if result else "FAIL"
    if not result:
        all_passed = False
    print(f"  [{status}] {check}")

print(f"\nOverall: {'ALL CHECKS PASSED' if all_passed else 'SOME CHECKS FAILED - REVIEW BEFORE SUBMITTING'}")

FINAL CONSISTENCY CHECKS:
  [PASS] Total loans = 12000
  [PASS] Total customers = 10000
  [PASS] Default rate around 20.5%
  [PASS] Gig worker highest default
  [PASS] BNPL highest product default
  [PASS] Social media highest channel
  [PASS] Tier 3 highest city default
  [PASS] Referral lowest channel default
  [PASS] Salaried lowest employment default
  [PASS] 5 risk_scored_portfolio columns

Overall: ALL CHECKS PASSED
